# ATS-Friendly Resume Generation System

This notebook implements a resume generation system that:
1. Takes a user profile (experience, bio, projects) and optionally a job description
2. Converts experience points to STAR/XYZ format
3. If job description is provided, matches and filters experience points
4. Selects top matching points for each section
5. Generates an ATS-friendly resume

## 1. Install Required Dependencies

In [10]:
# Install required packages
!pip install transformers datasets sentence-transformers nltk torch

## 2. Import Libraries and Set Up Environment

In [11]:
import json
import re
import torch
import os
import nltk
import numpy as np
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from sentence_transformers import SentenceTransformer, util

# Fix NLTK Data path issue
nltk_data_path = os.path.expanduser('~/nltk_data')
nltk.data.path.insert(0, nltk_data_path)

# Download required NLTK resources - try multiple times with explicit path
try:
    # Try standard download
    nltk.download('punkt')
    # Test if it works
    from nltk.tokenize import sent_tokenize
    sent_tokenize("This is a test. This is only a test.")
    print("NLTK punkt tokenizer working correctly.")
except LookupError:
    # Try with explicit path
    nltk.download('punkt', download_dir=nltk_data_path)
    print(f"Downloaded punkt to {nltk_data_path}")
    # Add to path again to be sure
    nltk.data.path.append(nltk_data_path)
    from nltk.tokenize import sent_tokenize
    
# Implement a fallback sentence tokenizer just in case
def safe_sentence_tokenize(text):
    """Safely tokenize text into sentences, with fallback if NLTK fails"""
    try:
        from nltk.tokenize import sent_tokenize
        return sent_tokenize(text)
    except (ImportError, LookupError):
        # Simple fallback tokenizer using common sentence terminators
        print("Using fallback sentence tokenizer")
        # First replace common abbreviations to avoid splitting them
        text = text.replace("Mr.", "Mr")
        text = text.replace("Ms.", "Ms")
        text = text.replace("Dr.", "Dr")
        text = text.replace("e.g.", "eg")
        text = text.replace("i.e.", "ie")
        
        # Split on sentence terminators
        sentences = re.split(r'(?<=[.!?])\s+', text)
        
        # Clean up and restore abbreviations
        sentences = [s.strip() for s in sentences if s.strip()]
        sentences = [s.replace("Mr", "Mr.") for s in sentences]
        sentences = [s.replace("Ms", "Ms.") for s in sentences]
        sentences = [s.replace("Dr", "Dr.") for s in sentences]
        sentences = [s.replace("eg", "e.g.") for s in sentences]
        sentences = [s.replace("ie", "i.e.") for s in sentences]
        
        return sentences

# Check if MPS (Apple Silicon GPU) is available
if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = torch.device("mps")
    print(f"Using Apple Silicon GPU via MPS")
else:
    device = torch.device("cpu")
    print(f"Using CPU - MPS not available")
    
# Set lower batch sizes and optimize for CPU/MPS if needed
use_low_memory = True  # Set to True for machines with limited memory

Downloaded punkt to /Users/sudhan/nltk_data
Using Apple Silicon GPU via MPS


[nltk_data] Downloading package punkt to /Users/sudhan/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt to /Users/sudhan/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


## 3. Load Models (Optional)

Note: For simplicity and reliability, this version of the notebook primarily uses the simple template-based implementation rather than loading the T5 model, which requires more memory and can cause issues. If you want to use the T5 model, uncomment and run the cell below.

In [12]:
# Optional: Load the T5 model for resume generation with optimizations
# Uncomment only if you want to use the T5 model rather than the template approach
model_name = "nakamoto-yama/t5-resume-generation"
print(f"Loading tokenizer from {model_name}...")
tokenizer = AutoTokenizer.from_pretrained(model_name)

print(f"Loading T5 model...")
if use_low_memory:
    # Use 8-bit quantization to reduce memory usage
    from transformers import AutoConfig
    config = AutoConfig.from_pretrained(model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(
        model_name,
        config=config,
        torch_dtype=torch.float16 if device.type != "cpu" else torch.float32,
        low_cpu_mem_usage=True
    ).to(device)
else:
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)

# Load sentence transformer model for matching
print(f"Loading sentence transformer model...")
try:
    # Use a smaller sentence transformer model for semantic matching
    sentence_model = SentenceTransformer('all-MiniLM-L6-v2', device=device.type)
    print("Sentence transformer model loaded successfully")
except Exception as e:
    print(f"Error loading sentence transformer model: {str(e)}")
    print("Will use simple word-based matching instead")

Loading tokenizer from nakamoto-yama/t5-resume-generation...
Loading T5 model...
Loading sentence transformer model...
Sentence transformer model loaded successfully


## 4. Define Helper Functions

### 4.1 Experience Point Formatting Functions

In [13]:
def simple_star_format_converter(experience_point):
    """
    A simple rule-based approach to convert experience points to STAR format
    """
    # Check if already in STAR/XYZ format
    if any(keyword in experience_point.lower() for keyword in 
           ['situation', 'task', 'action', 'result', 'challenge', 'context']):
        return experience_point
    
    # Split into sentences using our safe tokenizer
    sentences = safe_sentence_tokenize(experience_point)
    
    if len(sentences) <= 1:
        # For short points, create a template-based STAR format
        return f"Situation/Task: Identified opportunity for improvement. Action: {experience_point} Result: Successfully delivered measurable outcomes."
    elif len(sentences) == 2:
        return f"Situation/Task: {sentences[0]} Action/Result: {sentences[1]}"
    elif len(sentences) >= 3:
        situation_task = " ".join(sentences[:1])
        action = " ".join(sentences[1:2])
        result = " ".join(sentences[2:])
        return f"Situation/Task: {situation_task} Action: {action} Result: {result}"
    
    # Fallback
    return experience_point

### 4.2 Job Description Matching Functions

In [14]:
def simple_matching_algorithm(experience_point, job_description):
    """
    A simple keyword-based matching algorithm that doesn't require sentence transformers
    """
    # Extract keywords from job description
    job_words = set(job_description.lower().split())
    exp_words = set(experience_point.lower().split())
    
    # Calculate simple overlap
    common_words = job_words.intersection(exp_words)
    
    # Calculate score based on word overlap
    if len(exp_words) == 0:
        return 0
    
    return len(common_words) / len(exp_words)

def calculate_match_score(experience_point, job_description):
    """
    Calculate semantic similarity score between experience point and job description
    """
    try:
        # Try using sentence transformer if available
        if 'sentence_model' in globals():
            # Encode the texts into embeddings
            exp_embedding = sentence_model.encode(experience_point, convert_to_tensor=True)
            
            # Break the job description into sentences for better matching
            job_sentences = safe_sentence_tokenize(job_description)
            job_embeddings = sentence_model.encode(job_sentences, convert_to_tensor=True)
            
            # Calculate cosine similarity between experience point and each job sentence
            similarities = util.cos_sim(exp_embedding, job_embeddings)[0]
            
            # Return the maximum similarity score
            return float(torch.max(similarities).item())
        else:
            # Fallback to simple word matching
            return simple_matching_algorithm(experience_point, job_description)
    except Exception as e:
        print(f"Error calculating match score: {str(e)}")
        # Fallback to simple text matching
        return simple_matching_algorithm(experience_point, job_description)

def filter_and_rank_experiences(experiences, job_description, top_n=5):
    """
    Filter and rank experience points based on their match with the job description
    """
    if not job_description:
        # If no job description provided, return all experiences
        return experiences
    
    # Calculate match score for each experience point
    scored_experiences = []
    print(f"Scoring {len(experiences)} experience points...")
    for i, exp in enumerate(experiences):
        if i % 2 == 0:  # Print progress every 2 items
            print(f"Progress: {i+1}/{len(experiences)}")
        score = calculate_match_score(exp, job_description)
        scored_experiences.append((exp, score))
    
    # Sort experiences by score in descending order
    scored_experiences.sort(key=lambda x: x[1], reverse=True)
    
    # Print out the scores for visibility
    print("\nExperience Match Scores:")
    for exp, score in scored_experiences:
        shortened_exp = exp[:50] + "..." if len(exp) > 50 else exp
        print(f"Score: {score:.2f} - {shortened_exp}")
    
    # Select top N experiences
    top_experiences = [exp for exp, score in scored_experiences[:top_n]]
    
    return top_experiences

### 4.3 Resume Generation Functions

In [15]:
def clean_json_string(json_str):
    """
    Clean a JSON string to make it valid
    """
    # Replace problematic characters
    json_str = json_str.replace('\\', '\\\\')
    json_str = re.sub(r'[\x00-\x1F\x7F]', '', json_str)  # Remove control characters
    
    # Add missing quotes around keys
    json_str = re.sub(r'([{,])\s*([a-zA-Z_][a-zA-Z0-9_]*)\s*:', r'\1"\2":', json_str)
    
    # Fix trailing commas
    json_str = re.sub(r',\s*}', '}', json_str)
    json_str = re.sub(r',\s*]', ']', json_str)
    
    return json_str

In [16]:
def generate_resume_simple(profile, job_description=None):
    """
    A simplified version of resume generation that doesn't use the T5 model
    """
    print("Using simplified resume generation...")
    
    # Create a safe copy of the profile
    profile_copy = json.loads(json.dumps(profile))
    
    # Filter and convert experience points
    if job_description:
        print("Matching experiences to job description...")
        for section in profile_copy.get('experience', []):
            # Score and sort points
            points = section.get('points', [])
            scored_points = [(p, simple_matching_algorithm(p, job_description)) for p in points]
            scored_points.sort(key=lambda x: x[1], reverse=True)
            # Select top 5 points
            section['points'] = [p for p, _ in scored_points[:5]]
            
        for project in profile_copy.get('projects', []):
            points = project.get('points', [])
            scored_points = [(p, simple_matching_algorithm(p, job_description)) for p in points]
            scored_points.sort(key=lambda x: x[1], reverse=True)
            project['points'] = [p for p, _ in scored_points[:3]]
    
    # Convert points to STAR format using the simplified converter
    print("Converting experience points to STAR format...")
    for section in profile_copy.get('experience', []):
        section['points'] = [simple_star_format_converter(p) for p in section.get('points', [])]
    
    for project in profile_copy.get('projects', []):
        project['points'] = [simple_star_format_converter(p) for p in project.get('points', [])]
    
    # Create resume JSON
    resume_json = {
        "resume": {
            "personal_info": {
                "name": profile_copy.get("name", "Your Name"),
                "email": profile_copy.get("email", "your.email@example.com"),
                "phone": profile_copy.get("phone", "123-456-7890"),
                "summary": profile_copy.get("bio", "Professional summary here.")
            },
            "skills": profile_copy.get("skills", []),
            "experience": [
                {
                    "title": exp.get("title", ""),
                    "company": exp.get("company", ""),
                    "duration": exp.get("duration", ""),
                    "highlights": exp.get("points", [])
                } for exp in profile_copy.get("experience", [])
            ],
            "projects": [
                {
                    "name": proj.get("name", ""),
                    "duration": proj.get("duration", ""),
                    "highlights": proj.get("points", [])
                } for proj in profile_copy.get("projects", [])
            ]
        }
    }
    
    return resume_json

## 5. Example Usage

In [17]:
# Example user profile data
sample_profile = {
    "name": "John Doe",
    "email": "john.doe@example.com",
    "phone": "123-456-7890",
    "bio": "Experienced software engineer with a passion for building scalable web applications and cloud solutions.",
    "skills": ["Python", "JavaScript", "React", "AWS", "Docker", "Machine Learning", "SQL", "MongoDB"],
    "experience": [
        {
            "title": "Senior Software Engineer",
            "company": "Tech Solutions Inc.",
            "duration": "2020-Present",
            "points": [
                "Led a team of 5 engineers in developing a microservices architecture that improved system scalability by 200%",
                "Implemented CI/CD pipelines that reduced deployment time from 2 days to 30 minutes",
                "Designed and developed RESTful APIs consumed by the front-end team and external partners",
                "Mentored junior developers and conducted code reviews to maintain code quality",
                "Optimized database queries that reduced application response time by 40%"
            ]
        },
        {
            "title": "Software Developer",
            "company": "InnovateTech",
            "duration": "2018-2020",
            "points": [
                "Developed and maintained features for a SaaS product with over a million users",
                "Created responsive UI components using React and Redux",
                "Implemented authentication and authorization systems using OAuth 2.0",
                "Built data visualization dashboards that improved user engagement by 25%",
                "Refactored legacy codebase that improved maintainability and reduced bugs by 30%"
            ]
        }
    ],
    "projects": [
        {
            "name": "AI-Powered Content Recommendation Engine",
            "duration": "2022",
            "points": [
                "Developed a machine learning model that improved content recommendations by 45%",
                "Processed and analyzed large datasets using Python and Pandas",
                "Deployed the solution on AWS using Docker containers and Kubernetes"
            ]
        },
        {
            "name": "Open-Source Task Management Tool",
            "duration": "2019-2021",
            "points": [
                "Built a React-based front-end with a Node.js backend",
                "Implemented real-time collaboration features using WebSockets",
                "Created a plugin system that allowed for easy extension of functionality"
            ]
        }
    ]
}

In [18]:
# Example job description
sample_job_description = """
Senior Full Stack Developer

About Us:
We are a fast-growing tech company building innovative solutions for the financial sector. Our products serve millions of users globally, and we're looking for talented developers to join our team.

Job Description:
We are seeking a Senior Full Stack Developer to join our engineering team. You will be responsible for designing, developing, and maintaining web applications and services. The ideal candidate has a strong background in both front-end and back-end technologies and experience with cloud infrastructure.

Responsibilities:
- Design and develop scalable, maintainable, and efficient code
- Build RESTful APIs and microservices
- Implement responsive user interfaces using modern front-end frameworks
- Work with databases and optimize data access and storage
- Participate in code reviews and mentor junior developers
- Collaborate with product managers and UX designers
- Deploy and monitor applications in cloud environments

Requirements:
- 5+ years of software development experience
- Strong proficiency in JavaScript/TypeScript, React, and Node.js
- Experience with SQL and NoSQL databases
- Familiarity with cloud services (AWS, Azure, or GCP)
- Knowledge of containerization and orchestration (Docker, Kubernetes)
- Understanding of CI/CD pipelines and DevOps practices
- Excellent problem-solving and communication skills

Nice to have:
- Experience with Python or other backend languages
- Knowledge of machine learning or data analysis
- Contributions to open-source projects
"""

In [19]:
# Generate resume without job description (using all experience points)
resume_without_job = generate_resume_simple(sample_profile)
print(json.dumps(resume_without_job, indent=2))

Using simplified resume generation...
Converting experience points to STAR format...
Using fallback sentence tokenizer
Using fallback sentence tokenizer
Using fallback sentence tokenizer
Using fallback sentence tokenizer
Using fallback sentence tokenizer
Using fallback sentence tokenizer
Using fallback sentence tokenizer
Using fallback sentence tokenizer
Using fallback sentence tokenizer
Using fallback sentence tokenizer
Using fallback sentence tokenizer
Using fallback sentence tokenizer
Using fallback sentence tokenizer
Using fallback sentence tokenizer
Using fallback sentence tokenizer
Using fallback sentence tokenizer
{
  "resume": {
    "personal_info": {
      "name": "John Doe",
      "email": "john.doe@example.com",
      "phone": "123-456-7890",
      "summary": "Experienced software engineer with a passion for building scalable web applications and cloud solutions."
    },
    "skills": [
      "Python",
      "JavaScript",
      "React",
      "AWS",
      "Docker",
      "Ma

In [20]:
# Generate resume with job description (filtering experience points)
resume_with_job = generate_resume_simple(sample_profile, sample_job_description)
print(json.dumps(resume_with_job, indent=2))

Using simplified resume generation...
Matching experiences to job description...
Converting experience points to STAR format...
Using fallback sentence tokenizer
Using fallback sentence tokenizer
Using fallback sentence tokenizer
Using fallback sentence tokenizer
Using fallback sentence tokenizer
Using fallback sentence tokenizer
Using fallback sentence tokenizer
Using fallback sentence tokenizer
Using fallback sentence tokenizer
Using fallback sentence tokenizer
Using fallback sentence tokenizer
Using fallback sentence tokenizer
Using fallback sentence tokenizer
Using fallback sentence tokenizer
Using fallback sentence tokenizer
Using fallback sentence tokenizer
{
  "resume": {
    "personal_info": {
      "name": "John Doe",
      "email": "john.doe@example.com",
      "phone": "123-456-7890",
      "summary": "Experienced software engineer with a passion for building scalable web applications and cloud solutions."
    },
    "skills": [
      "Python",
      "JavaScript",
      "Rea

## 6. Function to Save Resume as Formatted Text or JSON

In [21]:
def save_resume(resume_json, output_format="json", filename="resume"):
    """
    Save the resume as a formatted text file or JSON
    """
    if output_format == "json":
        # Save as JSON
        try:
            # Make sure the JSON is valid before saving
            json_str = json.dumps(resume_json, indent=2)
            with open(f"{filename}.json", "w") as f:
                f.write(json_str)
            print(f"Resume saved as {filename}.json")
        except Exception as e:
            print(f"Error saving JSON: {str(e)}")
    else:
        # Save as formatted text
        try:
            # Extract resume data
            resume_data = resume_json.get("resume", {})
            personal_info = resume_data.get("personal_info", {})
            experiences = resume_data.get("experience", [])
            projects = resume_data.get("projects", [])
            skills = resume_data.get("skills", [])
            
            # Format resume text
            resume_text = f"{personal_info.get('name', 'Your Name')}\n"
            resume_text += f"{personal_info.get('email', '')} | {personal_info.get('phone', '')}\n\n"
            
            # Summary
            if 'summary' in personal_info:
                resume_text += "SUMMARY\n"
                resume_text += f"{personal_info['summary']}\n\n"
            
            # Skills
            resume_text += "SKILLS\n"
            resume_text += ", ".join(skills) + "\n\n"
            
            # Experience
            resume_text += "EXPERIENCE\n"
            for exp in experiences:
                resume_text += f"{exp.get('title', '')} - {exp.get('company', '')} ({exp.get('duration', '')})\n"
                for point in exp.get('highlights', []):
                    resume_text += f"• {point}\n"
                resume_text += "\n"
            
            # Projects
            resume_text += "PROJECTS\n"
            for project in projects:
                resume_text += f"{project.get('name', '')} ({project.get('duration', '')})\n"
                for point in project.get('highlights', []):
                    resume_text += f"• {point}\n"
                resume_text += "\n"
            
            # Save to file
            with open(f"{filename}.txt", "w") as f:
                f.write(resume_text)
            print(f"Resume saved as {filename}.txt")
            
        except Exception as e:
            print(f"Error formatting resume: {e}")
            # Fallback to JSON if formatting fails
            try:
                with open(f"{filename}.json", "w") as f:
                    json.dump(resume_json, f, indent=2)
                print(f"Resume saved as {filename}.json instead")
            except Exception as e2:
                print(f"Even JSON saving failed: {str(e2)}")

In [22]:
# Save the resume in different formats
save_resume(resume_with_job, output_format="json", filename="resume_json")
save_resume(resume_with_job, output_format="text", filename="resume_text")

Resume saved as resume_json.json
Resume saved as resume_text.txt


## 7. User Input Functions

The following functions allow users to input their own profile and job description interactively:

In [ ]:
def get_user_profile():
    """
    Interactive function to collect user profile information
    """
    profile = {}
    
    # Personal information
    profile["name"] = input("Enter your full name: ")
    profile["email"] = input("Enter your email: ")
    profile["phone"] = input("Enter your phone number: ")
    profile["bio"] = input("Enter a short professional summary: ")
    
    # Skills
    skills_input = input("Enter your skills (comma-separated): ")
    profile["skills"] = [skill.strip() for skill in skills_input.split(",")]
    
    # Experience
    experience = []
    while True:
        exp = {}
        exp["title"] = input("\nEnter job title (or leave empty to stop adding experiences): ")
        if not exp["title"]:
            break
        
        exp["company"] = input("Enter company name: ")
        exp["duration"] = input("Enter duration (e.g., 2020-2022): ")
        
        points = []
        print("Enter experience points (one per line, empty line to finish):")
        while True:
            point = input("- ")
            if not point:
                break
            points.append(point)
        
        exp["points"] = points
        experience.append(exp)
    
    profile["experience"] = experience
    
    # Projects
    projects = []
    while True:
        proj = {}
        proj["name"] = input("\nEnter project name (or leave empty to stop adding projects): ")
        if not proj["name"]:
            break
        
        proj["duration"] = input("Enter project duration (e.g., 2021-2022): ")
        
        points = []
        print("Enter project points (one per line, empty line to finish):")
        while True:
            point = input("- ")
            if not point:
                break
            points.append(point)
        
        proj["points"] = points
        projects.append(proj)
    
    profile["projects"] = projects
    
    return profile

def get_job_description():
    """
    Interactive function to collect job description
    """
    print("\nEnter job description (paste text, then enter 'END' on a new line when finished):")
    lines = []
    while True:
        line = input()
        if line == "END":
            break
        lines.append(line)
    
    return "\n".join(lines)

In [ ]:
# Uncomment and run this cell to use interactive input
"""
# Get user profile
print("Please enter your profile information:")
user_profile = get_user_profile()

# Ask if user wants to provide job description
use_job = input("\nDo you want to provide a job description for matching? (yes/no): ").lower() == "yes"

job_description = None
if use_job:
    job_description = get_job_description()

# Generate resume - use simple method for reliability
print("\nGenerating resume...")
resume = generate_resume_simple(user_profile, job_description)

# Save resume
output_format = input("\nSave as JSON or text? (json/text): ").lower()
filename = input("Enter filename (without extension): ")
save_resume(resume, output_format=output_format, filename=filename)
"""

## 8. Import Profiles from External Sources

You can load your profile data from various sources:

In [ ]:
def load_profile_from_json(filename):
    """
    Load user profile from a JSON file
    """
    try:
        with open(filename, 'r') as f:
            profile_str = f.read()
            # Clean the JSON string
            profile_str = clean_json_string(profile_str)
            profile = json.loads(profile_str)
        print(f"Successfully loaded profile from {filename}")
        return profile
    except json.JSONDecodeError as e:
        print(f"Error parsing JSON: {str(e)}")
        print("Please check your JSON file for syntax errors.")
        return None
    except Exception as e:
        print(f"Error loading profile: {e}")
        return None

In [ ]:
# Example of loading a profile from a JSON file
# Uncomment to use
"""
# Save the sample profile to a JSON file first
with open("sample_profile.json", "w") as f:
    json.dump(sample_profile, f, indent=2)

# Then load it
loaded_profile = load_profile_from_json("sample_profile.json")
if loaded_profile:
    # Generate a resume using the loaded profile
    loaded_resume = generate_resume_simple(loaded_profile, sample_job_description)
    save_resume(loaded_resume, output_format="text", filename="loaded_resume")
"""

## 9. Complete Resume Generation Workflow

In [ ]:
def resume_generation_workflow():
    """
    Complete resume generation workflow
    """
    print("===== ATS-FRIENDLY RESUME GENERATION SYSTEM =====")
    print("\nChoose an option:")
    print("1. Enter profile information interactively")
    print("2. Load profile from JSON file")
    print("3. Use sample profile (for testing)")
    
    choice = input("Enter your choice (1/2/3): ")
    
    # Get user profile
    if choice == "1":
        profile = get_user_profile()
    elif choice == "2":
        filename = input("Enter JSON file path: ")
        profile = load_profile_from_json(filename)
        if not profile:
            print("Could not load profile. Exiting.")
            return
    elif choice == "3":
        profile = sample_profile
        print("Using sample profile.")
    else:
        print("Invalid choice. Exiting.")
        return
    
    # Ask about job description
    print("\nDo you want to:")
    print("1. Use a job description for matching experiences")
    print("2. Generate resume without job description matching")
    print("3. Use sample job description (for testing)")
    
    job_choice = input("Enter your choice (1/2/3): ")
    
    # Get job description
    if job_choice == "1":
        job_description = get_job_description()
    elif job_choice == "2":
        job_description = None
    elif job_choice == "3":
        job_description = sample_job_description
        print("Using sample job description.")
    else:
        print("Invalid choice. Using no job description.")
        job_description = None
    
    # Generate resume
    print("\nGenerating resume...")
    try:
        resume = generate_resume_simple(profile, job_description)
    except Exception as e:
        print(f"Error in resume generation: {str(e)}")
        print("Falling back to basic template...")
        # Create a minimal resume
        resume = {
            "resume": {
                "personal_info": {
                    "name": profile.get("name", "Your Name"),
                    "email": profile.get("email", "your.email@example.com"),
                    "phone": profile.get("phone", "123-456-7890"),
                    "summary": profile.get("bio", "Professional summary here.")
                },
                "skills": profile.get("skills", []),
                "experience": []
            }
        }
    
    # Save options
    print("\nChoose output format:")
    print("1. JSON")
    print("2. Formatted text")
    print("3. Both")
    
    format_choice = input("Enter your choice (1/2/3): ")
    filename_base = input("Enter base filename (without extension): ")
    
    if format_choice == "1":
        save_resume(resume, output_format="json", filename=filename_base)
    elif format_choice == "2":
        save_resume(resume, output_format="text", filename=filename_base)
    elif format_choice == "3":
        save_resume(resume, output_format="json", filename=f"{filename_base}_json")
        save_resume(resume, output_format="text", filename=f"{filename_base}_text")
    else:
        print("Invalid choice. Saving as JSON.")
        save_resume(resume, output_format="json", filename=filename_base)
    
    print("\nResume generation complete!")
    return resume

In [ ]:
# Run the complete workflow
# Uncomment to use
# final_resume = resume_generation_workflow()

## 10. Tips for Using This System

1. **NLTK Troubleshooting**: If you encounter issues with NLTK, make sure the data is accessible from your Python environment. You can manually download the 'punkt' tokenizer with `nltk.download('punkt', download_dir='/your/preferred/path')` and then add that path to `nltk.data.path`.

2. **Memory Management**: This simplified version doesn't use the T5 model by default, which makes it much more reliable on machines with limited memory.

3. **Handling JSON Errors**: If you get JSON parsing errors, try using the `clean_json_string` function on your JSON file before loading it:
   ```python
   with open('your_file.json', 'r') as f:
       json_str = f.read()
   cleaned_str = clean_json_string(json_str)
   with open('fixed_file.json', 'w') as f:
       f.write(cleaned_str)
   ```

4. **Prepare detailed experience points**: The more detailed your experience points are, the better the conversion and job matching will be.

5. **Review and edit**: Always review and edit the generated resume, as the automated conversion may need refinement.

6. **Test different job descriptions**: Try matching your profile against different job descriptions to see how the system selects and prioritizes your experiences.

7. **Save multiple versions**: Generate and save multiple versions of your resume tailored to different job types.

8. **Optimize for ATS**: Remember that many companies use Applicant Tracking Systems (ATS) to filter resumes, so include relevant keywords from the job posting.